# Imports

In [1]:
import pandas as pd
import numpy as np

# Get and Format Data

In [2]:
def prepare_data(data, year):
    data = data.rename(columns=lambda x: x[2:] if x[1]=='_' else x)
    data['year'] = year
    data = data.sample(frac=1)
    return data

In [3]:
appalachia_train_sets = [prepare_data(pd.read_csv(f"datasets/post_pivot/plus_one/expanded_class/types/temperate/appalachia_train_{year}.csv"), year) for year in range(2022, 2025)]
appalachia_train = pd.concat(appalachia_train_sets, axis=0)

carpathians_train_sets = [prepare_data(pd.read_csv(f"datasets/post_pivot/plus_one/expanded_class/types/temperate/carpathians_train_{year}.csv"), year) for year in range(2022, 2025)]
carpathians_train = pd.concat(carpathians_train_sets, axis=0)

fareast_train_sets = [prepare_data(pd.read_csv(f"datasets/post_pivot/plus_one/expanded_class/types/temperate/fareast_train_{year}.csv"), year) for year in range(2022, 2025)]
fareast_train = pd.concat(fareast_train_sets, axis=0)

train_set = pd.concat([appalachia_train, carpathians_train, fareast_train], axis=0)
train_set = train_set.sample(frac=1)
train_set = train_set.sort_values(by='year')
print(train_set.head())
# print(train_set.groupby('year')['class'].count())

     system:index  NBR_delta_lag4  NBR_lag4  NDMI_delta_lag4  NDMI_lag4  \
5336     2_2336_0       -0.128555  0.166674        -0.097694   0.063969   
5876     2_2876_0        0.051183  0.126636         0.056974   0.014415   
1377     1_4377_0        0.080389  0.224498         0.030781   0.078316   
766      1_3766_0       -0.012710  0.140061         0.004036   0.019042   
4118     2_1118_0       -0.026023  0.304193        -0.080678   0.187088   

      NDVI_delta_lag4  NDVI_lag4  SR_B4_delta_lag4  SR_B4_lag4  \
5336        -0.124666   0.220173             344.5      8392.5   
5876         0.061920   0.256657             137.0      9470.0   
1377         0.030844   0.286030            -786.0      8397.0   
766         -0.021740   0.245523            -762.0      9563.0   
4118         0.009431   0.345407           -1041.0      8928.0   

      SR_B5_delta_lag4  ...  NDVI_lag0  SR_B4_lag0  SR_B5_lag0  SR_B6_lag0  \
5336           -3388.5  ...   0.344839      8048.0     16520.0     11922.0

In [4]:
print(len(train_set))

53985


In [5]:
all_features = train_set.columns.drop(['year', 'system:index', 'class', 'latitude', 'longitude', '.geo', 'year'])

recent_features = ['NBR_delta_lag1', 'NBR_lag1',
       'NDMI_delta_lag1', 'NDMI_lag1', 'NDVI_delta_lag1', 'NDVI_lag1',
       'SR_B4_delta_lag1', 'SR_B4_lag1', 'SR_B5_delta_lag1', 'SR_B5_lag1',
       'SR_B6_delta_lag1', 'SR_B6_lag1', 'SR_B7_delta_lag1', 'SR_B7_lag1',
       'NBR_lag0', 'NDMI_lag0', 'NDVI_lag0', 'SR_B4_lag0', 'SR_B5_lag0',
       'SR_B6_lag0', 'SR_B7_lag0']

In [6]:
from sklearn.model_selection import train_test_split

X = train_set[recent_features]
y = train_set['class']


# Evaluate Model on Training Data via Nested-CV

In [7]:
def custom_year_ts_split(df, year_col):
    """Custom CV splitter for our yearly data."""
    # Get unique years and sort them
    years = sorted(df[year_col].unique())
    
    # We need at least 2 years to do one split (Train Y1 -> Val Y2)
    for i in range(1, len(years)):
        # Training set: All years up to the current split
        train_indices = df[df[year_col].isin(years[:i])].index.values
        
        # Validation set: The very next year
        val_indices = df[df[year_col] == years[i]].index.values
        yield train_indices, val_indices

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate


In [9]:
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from xgboost import XGBClassifier

In [10]:
xgboost_param_grid = {
    'model__max_depth': Integer(3, 5),
    'model__min_child_weight': Integer(1, 200),
    'model__subsample': Real(0.7, 1),
    'model__colsample_bylevel': Real(0.5, 1),
    'model__colsample_bynode': Real(0.5, 1),
    'model__reg_lambda': Real(0, 10),
    'model__reg_alpha': Real(0, 10)
}

In [11]:
xgb_pipe = Pipeline([
    ('transformer', PowerTransformer('yeo-johnson')),
    ('model', XGBClassifier(objective='binary:logistic', random_state=1, learning_rate=0.3, tree_method='hist'))
])

tuner = BayesSearchCV(
    estimator=xgb_pipe,
    cv=5,
    n_iter=32,
    refit='f1',
    search_spaces=xgboost_param_grid,
    return_train_score=True,
    random_state=1,
    scoring='f1',
)

cv_iterator = custom_year_ts_split(train_set, 'year')

xgb_results = cross_validate(
    tuner,
    X, 
    y, 
    cv=cv_iterator, 
    scoring=['f1', 'precision', 'recall'], 
    return_estimator=True,
    return_train_score=True  # Crucial for detecting overfitting
)

KeyboardInterrupt: 

In [12]:
# Print inner-CV (5-fold) average train/test scores for each outer fold (using best inner params)
for outer_fold, outer_estimator in enumerate(xgb_results['estimator'], start=1):
    search = outer_estimator.named_steps['model'] if hasattr(outer_estimator, "named_steps") else outer_estimator
    inner_results = pd.DataFrame(search.cv_results_)
    best_idx = search.best_index_

    print(f"Outer Fold {outer_fold} Parameters - {search.best_params_}")
    mean_test = inner_results.loc[best_idx, "mean_test_score"]
    std_test = inner_results.loc[best_idx, "std_test_score"]
    print(f"Outer Fold {outer_fold} - Inner Mean Test Score:  {mean_test:.6f}")
    print(f"Outer Fold {outer_fold} - Inner Test Score STD:  {std_test:.6f}")

    if "mean_train_score" in inner_results.columns:
        mean_train = inner_results.loc[best_idx, "mean_train_score"]
        std_train = inner_results.loc[best_idx, "std_train_score"]
        print(f"Outer Fold {outer_fold} - Inner Mean Train Score: {mean_train:.6f}")
        print(f"Outer Fold {outer_fold} - Inner Train Score STD:  {std_test:.6f}")
    else:
        print(f"Outer Fold {outer_fold} - Inner Mean Train Score: not available (set return_train_score=True in BayesSearchCV)")

print('\nOverall Metrics')
print(f"Mean Test f1: {xgb_results['test_f1'].mean():.8f}")
print(f"Mean Test Precision: {xgb_results['test_precision'].mean():.8f}")
print(f"Mean Test Recall: {xgb_results['test_recall'].mean():.8f}")

print(f"Mean Train f1: {xgb_results['train_f1'].mean():.8f}")
print(f"Mean Train Precision: {xgb_results['train_precision'].mean():.8f}")
print(f"Mean Train Recall: {xgb_results['train_recall'].mean():.8f}")

print(f"Overfitting Ratio: {1-xgb_results['test_f1'].mean()/xgb_results['train_f1'].mean()}")

Outer Fold 1 Parameters - OrderedDict({'model__colsample_bylevel': 1.0, 'model__colsample_bynode': 1.0, 'model__max_depth': 5, 'model__min_child_weight': 1, 'model__reg_alpha': 0.0, 'model__reg_lambda': 0.0, 'model__subsample': 0.7})
Outer Fold 1 - Inner Mean Test Score:  0.989381
Outer Fold 1 - Inner Test Score STD:  0.000970
Outer Fold 1 - Inner Mean Train Score: 0.998861
Outer Fold 1 - Inner Train Score STD:  0.000970
Outer Fold 2 Parameters - OrderedDict({'model__colsample_bylevel': 1.0, 'model__colsample_bynode': 1.0, 'model__max_depth': 5, 'model__min_child_weight': 1, 'model__reg_alpha': 0.0, 'model__reg_lambda': 0.0, 'model__subsample': 0.7})
Outer Fold 2 - Inner Mean Test Score:  0.998737
Outer Fold 2 - Inner Test Score STD:  0.000681
Outer Fold 2 - Inner Mean Train Score: 0.999616
Outer Fold 2 - Inner Train Score STD:  0.000681

Overall Metrics
Mean Test f1: 0.99975285
Mean Test Precision: 1.00000000
Mean Test Recall: 0.99950584
Mean Train f1: 0.99975298
Mean Train Precision:

In [12]:
import shap
# Re-fit the tuner on full training data to get a final best estimator for interpretation
tuner.fit(X, y)
best_pipe = tuner.best_estimator_
model = best_pipe.named_steps['model']
explainer = shap.Explainer(model)
# Transform training data for the model component
X_trans = pd.DataFrame(
    best_pipe.named_steps['transformer'].transform(X),
    columns=recent_features,
    index=X.index,
)
shap_values = explainer(X_trans)
# visualize explanations
shap.plots.beeswarm(shap_values)
shap.plots.bar(shap_values)


KeyboardInterrupt: 

# Train Model

In [13]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

In [14]:
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer
from xgboost import XGBClassifier
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

In [18]:
xgboost_param_grid = {
    'max_depth': Integer(3, 5),
    'min_child_weight': Integer (1, 200),
    'subsample': Real(0.7, 1),
    'colsample_bylevel': Real(0.5, 1),
    'colsample_bynode': Real(0.5, 1),
    'reg_lambda': Real(0, 10),
    'reg_alpha': Real(0, 10)
}

In [19]:
stage_1_model = XGBClassifier(objective='binary:logistic', random_state=1, learning_rate=0.3, tree_method='hist')

tuner = BayesSearchCV(
    estimator=stage_1_model,
    cv=5,
    refit='f1',
    search_spaces=xgboost_param_grid,
    return_train_score=True,
    random_state=1,
    scoring='f1'
)

In [20]:
tuner.fit(X_train, y_train)

,estimator,"XGBClassifier...ree=None, ...)"
,search_spaces,"{'colsample_bylevel': Real(low=0.5,...m='normalize'), 'colsample_bynode': Real(low=0.5,...m='normalize'), 'max_depth': Integer(low=3...m='normalize'), 'min_child_weight': Integer(low=1...m='normalize'), ...}"
,optimizer_kwargs,None
,n_iter,50
,scoring,'f1'
,fit_params,None
,n_jobs,1
,n_points,1
,iid,'deprecated'
,refit,'f1'
,cv,5


In [27]:
params = tuner.best_params_
print(params)


OrderedDict({'colsample_bylevel': 0.6584961552734465, 'colsample_bynode': 1.0, 'max_depth': 5, 'min_child_weight': 1, 'reg_alpha': 10.0, 'reg_lambda': 10.0, 'subsample': 1.0})


In [28]:
stage_2_model = XGBClassifier(**params, objective='binary:logistic', random_state=1, learning_rate=0.001, early_stopping_rounds=50, eval_metric="logloss", tree_method='hist')

In [29]:
stage_2_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)], # Required for early stopping
    verbose=True,                     # Shows progress
)

[0]	validation_0-logloss:0.69256
[1]	validation_0-logloss:0.69196
[2]	validation_0-logloss:0.69136
[3]	validation_0-logloss:0.69075
[4]	validation_0-logloss:0.69015
[5]	validation_0-logloss:0.68957
[6]	validation_0-logloss:0.68899
[7]	validation_0-logloss:0.68839
[8]	validation_0-logloss:0.68779
[9]	validation_0-logloss:0.68719
[10]	validation_0-logloss:0.68660
[11]	validation_0-logloss:0.68601
[12]	validation_0-logloss:0.68541
[13]	validation_0-logloss:0.68483
[14]	validation_0-logloss:0.68424
[15]	validation_0-logloss:0.68365
[16]	validation_0-logloss:0.68306
[17]	validation_0-logloss:0.68246
[18]	validation_0-logloss:0.68188
[19]	validation_0-logloss:0.68129
[20]	validation_0-logloss:0.68071
[21]	validation_0-logloss:0.68013
[22]	validation_0-logloss:0.67954
[23]	validation_0-logloss:0.67896
[24]	validation_0-logloss:0.67838
[25]	validation_0-logloss:0.67781
[26]	validation_0-logloss:0.67723
[27]	validation_0-logloss:0.67665
[28]	validation_0-logloss:0.67608
[29]	validation_0-loglos

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,0.6584961552734465
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,1.0
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor

In [30]:
best_round = stage_2_model.best_iteration
print(best_round)
print(stage_2_model.best_score)


99
0.6379484380352571


In [31]:
final_model = XGBClassifier(**params, objective='binary:logistic', random_state=1, learning_rate=0.001, n_estimators=best_round, tree_method='hist')
final_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,0.6584961552734465
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,1.0
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegress

In [32]:
print(f1_score(y, final_model.predict(X)))

0.8750846375577953


In [33]:
import shap
explainer = shap.Explainer(final_model['model'])
# Transform training data for the model component
X_trans = pd.DataFrame(
    final_model.named_steps['transformer'].transform(X_train),
    columns=recent_features,
    index=X_train.index,
)
shap_values = explainer(X_trans)
# visualize explanations
shap.plots.beeswarm(shap_values)
shap.plots.bar(shap_values)


TypeError: 'XGBClassifier' object is not subscriptable

In [26]:
print(len(X))

53985


# Test On Holdout Set From the Same Region

In [34]:
appalachia_test = prepare_data(pd.read_csv(f"datasets/post_pivot/plus_one/expanded_class/types/temperate/appalachia_test_2025.csv"), 2025)
carpathians_test = prepare_data(pd.read_csv("datasets/post_pivot/plus_one/expanded_class/types/temperate/carpathians_test_2025.csv"), 2025)
fareast_test = prepare_data(pd.read_csv("datasets/post_pivot/plus_one/expanded_class/types/temperate/fareast_test_2025.csv"), 2025)

print(len(appalachia_test))
print(len(carpathians_test))
print(len(fareast_test))


6000
6000
5999


In [35]:
X_test_appalachia = appalachia_test[recent_features]
y_test_appalachia = appalachia_test['class']
X_test_carpathians = carpathians_test[recent_features]
y_test_carpathians = carpathians_test['class']
X_test_fareast = fareast_test[recent_features]
y_test_fareast = fareast_test['class']
print(y_test_carpathians.value_counts())

class
0    3000
1    3000
Name: count, dtype: int64


In [36]:
from sklearn.metrics import precision_score, recall_score

print("appalachia Test")
y_test_appalachia_pred = final_model.predict(X_test_appalachia)
print(f"f1: {f1_score(y_test_appalachia, y_test_appalachia_pred)}")
print(f"precision: {precision_score(y_test_appalachia, y_test_appalachia_pred)}")
print(f"recall: {recall_score(y_test_appalachia, y_test_appalachia_pred)}")

print("fareast Test")
y_test_fareast_pred = final_model.predict(X_test_fareast)
print(f"f1: {f1_score(y_test_fareast, y_test_fareast_pred)}")
print(f"precision: {precision_score(y_test_fareast, y_test_fareast_pred)}")
print(f"recall: {recall_score(y_test_fareast, y_test_fareast_pred)}")

print("carpathians Test")
y_test_carpathians_pred = final_model.predict(X_test_carpathians)
print(f"f1: {f1_score(y_test_carpathians, y_test_carpathians_pred)}")
print(f"precision: {precision_score(y_test_carpathians, y_test_carpathians_pred)}")
print(f"recall: {recall_score(y_test_carpathians, y_test_carpathians_pred)}")

print("All Together Test")
X_test = pd.concat([X_test_appalachia, X_test_fareast, X_test_carpathians])
y_test = pd.concat([y_test_appalachia, y_test_fareast, y_test_carpathians])
y_test_pred = final_model.predict(X_test)
print(f"f1: {f1_score(y_test, y_test_pred)}")
print(f"precision: {precision_score(y_test, y_test_pred)}")
print(f"recall: {recall_score(y_test, y_test_pred)}")

appalachia Test
f1: 0.8766265854060287
precision: 0.8664929990231195
recall: 0.887
fareast Test
f1: 0.8711894273127754
precision: 0.9241121495327103
recall: 0.824
carpathians Test
f1: 0.8589647411852963
precision: 0.9819897084048027
recall: 0.7633333333333333
All Together Test
f1: 0.8693055392903151
precision: 0.9189155731616737
recall: 0.8247777777777778


# Test on New Regions

In [37]:
valdivian_test = prepare_data(pd.read_csv(f"datasets/post_pivot/plus_one/expanded_class/types/temperate/valdivian_test_2025.csv"), 2025)

In [38]:
X_test_valdivian = valdivian_test[recent_features]
y_test_valdivian = valdivian_test['class']
print(y_test_valdivian.value_counts())

class
0    3000
1    3000
Name: count, dtype: int64


In [39]:
print("Valdivian Test")
y_test_valdivian_pred = final_model.predict(X_test_valdivian)
print(f"f1: {f1_score(y_test_valdivian, y_test_valdivian_pred)}")
print(f"precision: {precision_score(y_test_valdivian, y_test_valdivian_pred)}")
print(f"recall: {recall_score(y_test_valdivian, y_test_valdivian_pred)}")


Valdivian Test
f1: 0.7708830548926014
precision: 0.9556213017751479
recall: 0.646


Temperate model succeeds, though not quite as good on the Valdivian...

In [40]:
print(list(final_model.predict_proba(X_test_valdivian)))

[array([0.5348373, 0.4651627], dtype=float32), array([0.52845216, 0.47154784], dtype=float32), array([0.466958, 0.533042], dtype=float32), array([0.5351397 , 0.46486032], dtype=float32), array([0.5295844, 0.4704156], dtype=float32), array([0.50970477, 0.49029523], dtype=float32), array([0.45698893, 0.54301107], dtype=float32), array([0.54175174, 0.45824823], dtype=float32), array([0.4590146, 0.5409854], dtype=float32), array([0.46313173, 0.5368683 ], dtype=float32), array([0.46612704, 0.53387296], dtype=float32), array([0.50328565, 0.49671438], dtype=float32), array([0.53259957, 0.46740046], dtype=float32), array([0.47180474, 0.52819526], dtype=float32), array([0.46032637, 0.5396736 ], dtype=float32), array([0.45702922, 0.5429708 ], dtype=float32), array([0.5287869, 0.4712131], dtype=float32), array([0.53706765, 0.46293235], dtype=float32), array([0.533417  , 0.46658298], dtype=float32), array([0.5341897, 0.4658103], dtype=float32), array([0.50327   , 0.49673003], dtype=float32), array

In [45]:
final_model.save_model("../app/ml_models/xgb_temperate.json")